# CDS in-situ surface-land — demo

End-to-end demo for the CDS `cds_insitu_land` adapter, showing the `.env` → `CDSSource` → `CDSInsituArchive` pipeline and a GeoParquet round trip.

Running this notebook requires live CDS credentials in a project-root `.env`:

```
CDSAPI_URL=https://cds.climate.copernicus.eu/api
CDSAPI_KEY=<uid>:<api-key>
```

Dataset: **Global land surface atmospheric variables from 1755 to present (CDS `insitu-observations-surface-land`)**. The demo filters to the Iberian peninsula client-side; CDS in-situ does not accept a server-side `area` key.

## 1. Set up the adapter and archive

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt

from xr_toolz.data import CDSInsituArchive, CDSSource
from xr_toolz.types import BBox

In [ ]:
scratch_root = Path.cwd() / "scratch" / "cds_insitu_demo"
archive = CDSInsituArchive(
    root=scratch_root,
    preset='cds_insitu_land',
    source=CDSSource(),
    time_aggregation="daily",
)
archive.preset_root

## 2. Sync a one-year window

The archive chunks by year and writes GeoParquet; re-running is a no-op for already-completed years.

In [ ]:
fresh = archive.sync(
    "2020-01-01",
    "2020-12-31",
    bbox=BBox(lon_min=-10.0, lon_max=3.5, lat_min=35.5, lat_max=44.0)  # Iberia,
)
len(fresh)

## 3. Read back the GeoParquet archive

In [ ]:
gdf = archive.load()
gdf.head()

In [ ]:
gdf.groupby("variable").size().sort_values(ascending=False)

## 4. Plot station locations

In [ ]:
stations = archive.load_stations()
fig, ax = plt.subplots(figsize=(8, 5))
stations.plot(ax=ax, markersize=10, color="tab:red")
ax.set_xlabel("lon")
ax.set_ylabel("lat")
ax.set_title(stations.shape[0], "land stations in window")
plt.show()

## 5. Per-station time series

In [ ]:
ds = archive.load_dataset()
ds